In [1]:
# import necessary libraries and packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             roc_auc_score, f1_score, precision_score, recall_score)
from sklearn.base import BaseEstimator, ClassifierMixin
from datetime import datetime
import os

# pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
torch.use_deterministic_algorithms(True)  # will error if a nondeterministic op is used
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# optional but often helps remove GPU variability
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


In [3]:
# create timestamp folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_dir = f"model_results/nn_fusion_clf_results_cross_val/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

In [4]:
# option for saving ALL predictions as a pkl file (the file likely will be around 2 gb)
save_predictions_pkl = False

In [5]:
# set seed
seed = 42
import random
random.seed(seed)
np.random.seed(seed)

# set seed for pytorch
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# ensure deterministic behavior for cuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# pytorch generator for dataloader
torch_generator = torch.Generator().manual_seed(seed)

os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [6]:
# read in main csv file to get target values
main_df = pd.read_csv("../data/unified_features_final_subset_og_seqs.csv",low_memory = False) 
main_df = main_df.rename(columns = {'protein_seq': 'seq'})

In [7]:
# embeddings file paths
embedding_paths = {'seqvec': r"../data/embeddings/bio_embeddings/seqvec_embeddings.csv", 
                   'bepler': r"../data/embeddings/bio_embeddings/bepler_embeddings.csv", 
                   'cpcprot': r"../data/embeddings/bio_embeddings/cpcprot_embeddings.csv", 
                   'esm': r"../data/embeddings/bio_embeddings/esm_embeddings.csv", 
                   'esm1b': r"../data/embeddings/bio_embeddings/esm1b_embeddings.csv",
                   'esm2': r"../data/embeddings/bio_embeddings/esm2_embeddings.csv",
                   'plusrnn': r"../data/embeddings/bio_embeddings/plusrnn_embeddings.csv",
                   'prottrans_bert_bfd': r"../data/embeddings/bio_embeddings/prottrans_bert_bfd.csv"
                }

In [8]:
# define feature columns
og_feature_cols = [
    'fraction_A','fraction_C','fraction_D','fraction_E','fraction_F','fraction_G','fraction_H','fraction_I','fraction_K','fraction_L','fraction_M',
    'fraction_N','fraction_P','fraction_Q','fraction_R','fraction_S','fraction_T','fraction_V','fraction_W','fraction_Y',
    'fraction_group_ILMV','fraction_group_RK','fraction_group_DE','fraction_group_GS','fraction_group_YFW',
    'NCPR','FCR','fraction_disorder_promoting','mean_hydropathy',
    'frac_disorder','omega_STNQCH','kappa_STNQCH_ILMV','kappa_STNQCH_RK','kappa_STNQCH_ED',
    'kappa_STNQCH_FWY','kappa_STNQCH_A','kappa_STNQCH_P','kappa_STNQCH_G','omega_ILMV','kappa_ILMV_RK','kappa_ILMV_ED',
    'kappa_ILMV_FWY','kappa_ILMV_A','kappa_ILMV_P','kappa_ILMV_G','omega_RK','kappa_RK_ED','kappa_RK_FWY','kappa_RK_A',
    'kappa_RK_P','kappa_RK_G','omega_ED','kappa_ED_FWY','kappa_ED_A','kappa_ED_P','kappa_ED_G','omega_FWY',
    'kappa_FWY_A','kappa_FWY_P','kappa_FWY_G','omega_A','kappa_A_P','kappa_A_G','omega_P','kappa_P_G','omega_G',
    'fraction_R_of_RK','fraction_D_of_DE','fraction_S_of_SG','fraction_N_of_NQ','fraction_Y_of_YF','fraction_F_of_FW',
    'fraction_Y_of_YW','fraction_R_of_RQ','fraction_K_of_KQ','fraction_group_ILV','fraction_FYW_of_FYWILV','fraction_FYW_of_FYWR', 
]

new_feature_cols = [
    'lcr_residues',
    'radius_of_gyration', 'scaling_exponent', 'asphericity',
    'pslab_delta_g', 'pslab_saturation_mgml', 'finches_heterotypic_epsilon',
    'finches_homotypic_epsilon'
]

bryans_feature_cols = [ 
    'frac_polar', 'frac_hydrophobic', 'frac_tiny', 'frac_small', 
    'frac_aliphatic', 'charge_ratio', 'polar_nonpolar_ratio', 
    'molecular_weight', 'isoelectric_point', 'scd', 'charge_segregation', 
    'hydrophobic_moment', 'aromatic_clustering', 'frac_sticker', 'frac_spacer', 'sticker_spacer_ratio', 'max_sticker_spacing', 
    'meta_disorder_mean', 'meta_disorder_max'
]

feature_cols = og_feature_cols + new_feature_cols + bryans_feature_cols

# save list of features to txt file in timestamp folder
list_file = f"{output_dir}/feature_list.txt"

with open(list_file, 'w') as file:
    for feature in feature_cols:
        file.write(str(feature) + '\n')


In [9]:
# initialize list for storing model results
clf_list = []

In [10]:
# initialize dict for storing model predictions
trained_models_dict = {}
all_predictions_dict = {}

In [11]:
# initialize list of target columns
target_cols = [
    #"low_GFP_fraction_cells_with_condensates",
    #"low_SNAP_fraction_cells_with_condensates",
    "medium_GFP_fraction_cells_with_condensates"
    #"medium_SNAP_fraction_cells_with_condensates"
    #"high_GFP_fraction_cells_with_condensates",
    #"high_SNAP_fraction_cells_with_condensates"
]

In [12]:
# unimodal branch network for marginal representations
class unimodalBranch(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout_rate=0.2):
        
        super().__init__()

        # set seed
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            
        layers = []
        prev_dim = input_dim
        
        # create hidden layers with relu activation
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))  # batch normalization for stability
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))  # dropout for regularization
            
            prev_dim = h_dim
        
        self.net = nn.Sequential(*layers)
        self.output_dim = hidden_dims[-1] if hidden_dims else input_dim
    
    def forward(self, x):
        return self.net(x)

In [13]:
# fusion network for learning joint representations
class fusionNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims=[32, 16], output_dim=2, dropout_rate=0.2):
        
        super().__init__()

        # set seed
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            
        layers = []
        prev_dim = input_dim
        
        # create hidden layers
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            
            prev_dim = h_dim
        
        # final classification layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [14]:
class intermediateFusionNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, feature_dim, embedding_dim, lr=0.001, epochs=50, batch_size=32, dropout_rate=0.3, use_cross_attention = True, patience = 10, min_delta = 0.001, validation_split = 0.1):
        
        super().__init__()
        self.feature_dim = feature_dim
        self.embedding_dim = embedding_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.dropout_rate = dropout_rate
        self.use_cross_attention = use_cross_attention
        self.patience = patience
        self.min_delta = min_delta
        self.validation_split = validation_split

        self.train_loss_history = []
        self.val_loss_history = []

        # set seed for reproducibility
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        # early stopping tracking
        self.best_val_loss = float('inf')
        self.best_val_accuracy = 0
        self.best_model_state = None
        self.patience_counter = 0
        
        # device configuration
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # initialize networks
        self._init_networks()
        
        # move networks to device
        self.to(self.device)
        
        # training state
        self.is_fitted = False
        self.train_loss_history = []
        self.val_loss_history = []
    
    def _init_networks(self):
        # unimodal branch for biophysical features
        self.feature_net = unimodalBranch(
            input_dim=self.feature_dim, 
            hidden_dims=[32, 16], 
            dropout_rate=self.dropout_rate
        )
        
        # unimodal branch for protein embeddings
        self.embedding_net = unimodalBranch(
            input_dim=self.embedding_dim, 
            hidden_dims=[64, 32], 
            dropout_rate=self.dropout_rate
        )

        # add cross attention
        if self.use_cross_attention:

            # get actual output dims
            feature_output_dim = self.feature_net.output_dim
            embedding_output_dim = self.embedding_net.output_dim

            # project both to same dim
            self.target_attention_dim = 16
            
            # add projection layers
            self.embedding_projection = nn.Linear(embedding_output_dim, self.target_attention_dim)
            self.feature_projection = nn.Linear(feature_output_dim, self.target_attention_dim)
            
            self.cross_attention = nn.MultiheadAttention(
                embed_dim = self.target_attention_dim,
                num_heads = 4,
                batch_first = True,
                dropout = self.dropout_rate
            )

            # layer normalization after attention
            self.layer_norm1 = nn.LayerNorm(self.target_attention_dim)
            self.layer_norm2 = nn.LayerNorm(self.target_attention_dim)
            
        # fusion network takes concatenated marginal representations
        if self.use_cross_attention:
            fusion_input_dim = self.target_attention_dim * 2
        else:
            fusion_input_dim = self.feature_net.output_dim + self.embedding_net.output_dim
            
        self.fusion_net = fusionNetwork(
            input_dim=fusion_input_dim,
            hidden_dims=[16, 8],
            output_dim=2,  # binary classification
            dropout_rate=self.dropout_rate
        )

    def forward_with_attention(self, batch_features, batch_embeddings):
        # forward pass with cross attention integration

        # get raw marginal representations
        feature_marginal_raw = self.feature_net(batch_features)  # [batch, 32]
        embedding_marginal_raw = self.embedding_net(batch_embeddings)  # [batch, 64]
        
        # project to same dimension:
        feature_marginal = self.feature_projection(feature_marginal_raw)  # [batch, 32]
        embedding_marginal = self.embedding_projection(embedding_marginal_raw)  # [batch, 32]

        # reshape for attention
        feature_reshaped = feature_marginal.unsqueeze(1)
        embedding_reshaped = embedding_marginal.unsqueeze(1)

        # apply attention (features to embeddings)
        attended_features, _ = self.cross_attention(
            feature_reshaped,
            embedding_reshaped,
            embedding_reshaped,
            need_weights = False
        )

        attended_features = self.layer_norm1(feature_reshaped + attended_features)

        # apply attention (embeddings to features)
        attended_embeddings, _ = self.cross_attention(
            embedding_reshaped,
            feature_reshaped,
            feature_reshaped,
            need_weights = False
        )

        attended_embeddings = self.layer_norm2(embedding_reshaped + attended_embeddings)

        feature_marginal_attended = attended_features.squeeze(1)
        embedding_marginal_attended = attended_embeddings.squeeze(1)
        
        # concatenate and fuse
        fused = torch.cat([feature_marginal_attended, embedding_marginal_attended], dim=1)
        outputs = self.fusion_net(fused)

        return outputs, feature_marginal, embedding_marginal
        
    def to(self, device):
        self.device = device
        self.feature_net.to(device)
        self.embedding_net.to(device)
        self.fusion_net.to(device)

        if self.use_cross_attention:
            self.embedding_projection.to(device)
            self.feature_projection.to(device)
            self.cross_attention.to(device)
            self.layer_norm1.to(device)
            self.layer_norm2.to(device)

        return self
    
    def fit(self, x_features_train, x_embeddings_train, y_train,
        x_features_val=None, x_embeddings_val=None, y_val=None):

        if x_features_val is None or x_embeddings_val is None or y_val is None:
            raise ValueError("Provide x_features_val/x_embeddings_val/y_val to use premade splits.")

        x_features_train = np.asarray(x_features_train)
        x_embeddings_train = np.asarray(x_embeddings_train)
        y_train = np.asarray(y_train)

        x_features_val = np.asarray(x_features_val)
        x_embeddings_val = np.asarray(x_embeddings_val)
        y_val = np.asarray(y_val)


        if x_features_val is None:
            raise ValueError("Provide x_features_val/x_embeddings_val/y_val to use premade splits.")

        # convert training data to torch tensors
        x_features_train_tensor = torch.tensor(x_features_train, dtype=torch.float32)
        x_embeddings_train_tensor = torch.tensor(x_embeddings_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.long)
        
        # convert validation data to torch tensors
        x_features_val_tensor = torch.tensor(x_features_val, dtype=torch.float32)
        x_embeddings_val_tensor = torch.tensor(x_embeddings_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.long)
        
        # create training dataloader
        train_dataset = TensorDataset(x_features_train_tensor, x_embeddings_train_tensor, y_train_tensor)
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True, worker_init_fn=lambda worker_id: torch.manual_seed(seed + worker_id), generator = torch_generator, drop_last = True)
        
        # create validation dataloader
        val_dataset = TensorDataset(x_features_val_tensor, x_embeddings_val_tensor, y_val_tensor)
        val_dataloader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=lambda worker_id: torch.manual_seed(seed + worker_id), generator = torch_generator)
        
        # define loss function
        criterion = nn.CrossEntropyLoss()
        
        # collect all parameters
        params = list(self.feature_net.parameters()) + \
                list(self.embedding_net.parameters()) + \
                list(self.fusion_net.parameters())
        
        # add cross-attention parameters if used
        if self.use_cross_attention:
            params += list(self.embedding_projection.parameters())
            params += list(self.feature_projection.parameters())
            params += list(self.cross_attention.parameters())
            params += list(self.layer_norm1.parameters())
            params += list(self.layer_norm2.parameters())

        # define optimizer
        optimizer = optim.AdamW(params, lr=self.lr, weight_decay = 1e-4)

        # add learning rate scheduler
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max = self.epochs, eta_min = 1e-6
        )
        
        # reset early stopping tracking
        self.best_val_loss = float('inf')
        self.best_val_accuracy = 0
        self.patience_counter = 0
        self.best_model_state = None
        
        # training loop
        self.feature_net.train()
        self.embedding_net.train()
        self.fusion_net.train()
        if self.use_cross_attention:
            self.cross_attention.train()

        for epoch in range(self.epochs):
            # early stopping check 
            if self.patience_counter >= self.patience:
                print(f"  Early stopping triggered at epoch {epoch+1}")
                # Restore best model BEFORE breaking
                self._restore_best_model()
                break
            
            epoch_train_loss = 0.0
            
            for batch_features, batch_embeddings, batch_labels in train_dataloader:
                # move batch to device
                batch_features = batch_features.to(self.device)
                batch_embeddings = batch_embeddings.to(self.device)
                batch_labels = batch_labels.to(self.device)
                
                # zero the parameter gradients
                optimizer.zero_grad()
                
                # forward pass with or without attention
                if self.use_cross_attention:
                    outputs, feature_marginal, embedding_marginal = self.forward_with_attention(
                        batch_features, batch_embeddings
                    )
                    
                    # calculate correlation between marginal representations
                    feature_norm = feature_marginal - feature_marginal.mean(dim=0)
                    embedding_norm = embedding_marginal - embedding_marginal.mean(dim=0)
                    
                    # compute correlation matrix
                    corr_matrix = torch.mm(feature_norm.t(), embedding_norm)
                    corr_matrix = corr_matrix / (feature_norm.shape[0] - 1 + 1e-8)
                    
                    # diversity loss: minimize absolute correlation
                    diversity_loss = torch.mean(torch.abs(corr_matrix))
                    
                    # calculate total loss
                    class_loss = criterion(outputs, batch_labels)
                    loss = class_loss + 0.01 * diversity_loss  # Weighted combination
                    
                else:
                    # original forward pass without attention
                    feature_marginal = self.feature_net(batch_features)
                    embedding_marginal = self.embedding_net(batch_embeddings)
                    
                    # concatenate marginal representations for fusion
                    fused = torch.cat([feature_marginal, embedding_marginal], dim=1)
                    
                    # forward pass through fusion network
                    outputs = self.fusion_net(fused)
                    
                    # calculate loss
                    loss = criterion(outputs, batch_labels)
                
                # backward pass and optimize
                loss.backward()

                # gradient clpping
                torch.nn.utils.clip_grad_norm_(params, max_norm = 1.0)
                
                optimizer.step()
                optimizer.zero_grad()
                
                epoch_train_loss += loss.item()
            
            # validation phase
            epoch_val_loss = 0.0
            self.feature_net.eval()
            self.embedding_net.eval()
            self.fusion_net.eval()
            if self.use_cross_attention:
                self.cross_attention.eval()
            
            with torch.no_grad():
                for batch_features, batch_embeddings, batch_labels in val_dataloader:
                    batch_features = batch_features.to(self.device)
                    batch_embeddings = batch_embeddings.to(self.device)
                    batch_labels = batch_labels.to(self.device)
                    
                    if self.use_cross_attention:
                        outputs, _, _ = self.forward_with_attention(batch_features, batch_embeddings)
                    else:
                        feature_marginal = self.feature_net(batch_features)
                        embedding_marginal = self.embedding_net(batch_embeddings)
                        fused = torch.cat([feature_marginal, embedding_marginal], dim=1)
                        outputs = self.fusion_net(fused)
                    
                    loss = criterion(outputs, batch_labels)
                    epoch_val_loss += loss.item()
            
            avg_train_loss = epoch_train_loss / len(train_dataloader)
            avg_val_loss = epoch_val_loss / len(val_dataloader)

            # record loss
            self.train_loss_history.append(avg_train_loss)
            self.val_loss_history.append(avg_val_loss)
            
            scheduler.step()
            
            # early stopping logic
            if avg_val_loss < self.best_val_loss:
                self.best_val_loss = avg_val_loss
                self.patience_counter = 0
                # save best model state
                self.best_model_state = {
                    'feature_net': self.feature_net.state_dict(),
                    'embedding_net': self.embedding_net.state_dict(),
                    'fusion_net': self.fusion_net.state_dict(),
                }
                if self.use_cross_attention:
                    self.best_model_state['cross_attention'] = self.cross_attention.state_dict()
                    self.best_model_state['layer_norm1'] = self.layer_norm1.state_dict()
                    self.best_model_state['layer_norm2'] = self.layer_norm2.state_dict()
            else:
                self.patience_counter += 1
            
            # set back to training mode for next epoch
            self.feature_net.train()
            self.embedding_net.train()
            self.fusion_net.train()
            if self.use_cross_attention:
                self.cross_attention.train()
            
            # print training progress every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f"  epoch {epoch + 1}/{self.epochs}, train loss: {avg_train_loss:.4f}, val loss: {avg_val_loss:.4f}, patience: {self.patience_counter}/{self.patience}")
        
        # if we finished all epochs without early stopping, restore best model
        if self.best_model_state is not None and self.patience_counter < self.patience:
            print("Training completed without early stopping - restoring best model")
            self._restore_best_model()
        
        self.is_fitted = True
        return self

    def _restore_best_model(self):
        """Restore the best model state saved during training"""
        if self.best_model_state is None:
            print("Warning: No best model state to restore")
            return
        
        print(f"  Restoring best model with validation loss: {self.best_val_loss:.4f}")
        
        self.feature_net.load_state_dict(self.best_model_state['feature_net'])
        self.embedding_net.load_state_dict(self.best_model_state['embedding_net'])
        self.fusion_net.load_state_dict(self.best_model_state['fusion_net'])
        
        if self.use_cross_attention:
            self.cross_attention.load_state_dict(self.best_model_state['cross_attention'])
            self.layer_norm1.load_state_dict(self.best_model_state['layer_norm1'])
            self.layer_norm2.load_state_dict(self.best_model_state['layer_norm2'])
            
    def predict(self, x_features, x_embeddings):
        """make class predictions"""
        proba = self.predict_proba(x_features, x_embeddings)
        return np.argmax(proba, axis=1)
    
    def predict_proba(self, x_features, x_embeddings):
        """predict class probabilities"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")

        x_features_tensor = torch.tensor(x_features, dtype=torch.float32).to(self.device)
        x_embeddings_tensor = torch.tensor(x_embeddings, dtype=torch.float32).to(self.device)

        self.feature_net.eval()
        self.embedding_net.eval()
        self.fusion_net.eval()
        if self.use_cross_attention:
            self.embedding_projection.eval()
            self.feature_projection.eval()
            self.cross_attention.eval()
            self.layer_norm1.eval()
            self.layer_norm2.eval()

        with torch.no_grad():
            if self.use_cross_attention:
                logits, _, _ = self.forward_with_attention(x_features_tensor, x_embeddings_tensor)
            else:
                feature_marginal = self.feature_net(x_features_tensor)
                embedding_marginal = self.embedding_net(x_embeddings_tensor)
                fused = torch.cat([feature_marginal, embedding_marginal], dim=1)
                logits = self.fusion_net(fused)

            probabilities = torch.softmax(logits, dim=1)

        return probabilities.cpu().numpy()
        
    def get_per_sample_losses(self, x_features, x_embeddings, y_true):

        # set all PyTorch modules to eval mode
        self.feature_net.eval()
        self.embedding_net.eval()
        self.fusion_net.eval()
        if self.use_cross_attention:
            self.cross_attention.eval()
            self.embedding_projection.eval()
            self.feature_projection.eval()
            self.layer_norm1.eval()
            self.layer_norm2.eval()
            
        with torch.no_grad():
            # convert to tensors
            x_features_tensor = torch.tensor(x_features, dtype=torch.float32).to(self.device)
            x_embeddings_tensor = torch.tensor(x_embeddings, dtype=torch.float32).to(self.device)
            
            # get logits based on whether attention is used
            if self.use_cross_attention:
                logits, _, _ = self.forward_with_attention(x_features_tensor, x_embeddings_tensor)
            else:
                feature_marginal = self.feature_net(x_features_tensor)
                embedding_marginal = self.embedding_net(x_embeddings_tensor)
                fused = torch.cat([feature_marginal, embedding_marginal], dim=1)
                logits = self.fusion_net(fused)
            
            # per-sample loss (cross entropy without reduction)
            criterion = nn.CrossEntropyLoss(reduction='none')
            y_true_tensor = torch.tensor(y_true, dtype=torch.long).to(self.device)
            losses = criterion(logits, y_true_tensor)
            
        return losses.cpu().numpy()
    
    def get_marginal_representations(self, x_features, x_embeddings):
        # extract marginal representations from unimodal branches.
        
        x_features_tensor = torch.tensor(x_features, dtype=torch.float32).to(self.device)
        x_embeddings_tensor = torch.tensor(x_embeddings, dtype=torch.float32).to(self.device)
        
        self.feature_net.eval()
        self.embedding_net.eval()
        if self.use_cross_attention:
            self.embedding_projection.eval()
            self.feature_projection.eval()
            self.cross_attention.eval()
        
        with torch.no_grad():
            if self.use_cross_attention:
                # get attended marginals
                feature_marginal_raw = self.feature_net(x_features_tensor)
                embedding_marginal_raw = self.embedding_net(x_embeddings_tensor)
                
                # project to same dimension
                feature_marginal = self.feature_projection(feature_marginal_raw)
                embedding_marginal = self.embedding_projection(embedding_marginal_raw)
                
                # apply attention
                feature_reshaped = feature_marginal.unsqueeze(1)
                embedding_reshaped = embedding_marginal.unsqueeze(1)
                
                attended_features, _ = self.cross_attention(
                    feature_reshaped, embedding_reshaped, embedding_reshaped, need_weights=False
                )
                attended_embeddings, _ = self.cross_attention(
                    embedding_reshaped, feature_reshaped, feature_reshaped, need_weights=False
                )
                
                feature_marginal = attended_features.squeeze(1).cpu().numpy()  # 32D
                embedding_marginal = attended_embeddings.squeeze(1).cpu().numpy()  # 32D
            else:
                feature_marginal = self.feature_net(x_features_tensor).cpu().numpy()
                embedding_marginal = self.embedding_net(x_embeddings_tensor).cpu().numpy()
        
        return feature_marginal, embedding_marginal

In [15]:
def load_embedding_data(embedding_file, seq_col='seq'):
    embedding_df = pd.read_csv(embedding_file)
    
    # add seq col to target cols to ensure all are included in the merge
    columns_to_include = [seq_col, 'protein_id'] + target_cols
    
    # merge all data
    embedding_df = pd.merge(embedding_df, main_df[feature_cols + columns_to_include], on=seq_col, how='left')
    
    return embedding_df

In [16]:
def extract_modality_features(df, target_col, seq_col='seq'):

    # get subset of data where target col is not na
    subset_df = df.dropna(subset=target_col)
    subset_df = subset_df.dropna(subset = ['protein_id'])
    
    columns_to_exclude = [seq_col] + target_cols + ['base_split', 'protein_id']
    
    # separate embedding features and biophysical features
    all_features = [col for col in subset_df.columns if col not in columns_to_exclude]
    embedding_cols = [col for col in all_features if col not in feature_cols]
    
    # extract features and target
    x_embeddings = subset_df[embedding_cols].values
    x_features = subset_df[feature_cols].values
    y = subset_df[target_col].values
    groups = subset_df['protein_id'].values
    
    return x_features, x_embeddings, y, groups, subset_df

In [17]:
def assert_no_group_overlap(idx_a, idx_b, groups, label_a, label_b):
    g = np.asarray(groups)
    overlap = np.intersect1d(g[idx_a], g[idx_b])
    if overlap.size > 0:
        raise AssertionError(f"Group leakage between {label_a} and {label_b}: {overlap[:10].tolist()}")

In [18]:
def prepare_data_modalities_for_fold(x_features, x_embeddings, y, groups, train_idx, test_idx, val_size = 0.15, random_state = 42):
    # clean NaNs
    x_features = np.nan_to_num(np.asarray(x_features), nan=0.0)
    x_embeddings = np.nan_to_num(np.asarray(x_embeddings), nan=0.0)
    y = np.asarray(y)
    groups = np.asarray(groups)

    gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    train_sub_idx, val_sub_idx = next(gss.split(x_features[train_idx], y[train_idx], groups=groups[train_idx]))

    final_train_idx = train_idx[train_sub_idx]
    final_val_idx = train_idx[val_sub_idx]

    assert_no_group_overlap(final_train_idx, final_val_idx, groups, "train", "val")
    assert_no_group_overlap(final_train_idx, test_idx, groups, "train", "test")
    assert_no_group_overlap(final_val_idx, test_idx, groups, "val", "test")

    xF_train, xF_val, xF_test = x_features[final_train_idx], x_features[final_val_idx], x_features[test_idx]
    xE_train, xE_val, xE_test = x_embeddings[final_train_idx], x_embeddings[final_val_idx], x_embeddings[test_idx]
    y_train, y_val, y_test = y[final_train_idx], y[final_val_idx], y[test_idx]

    feature_scaler, embedding_scaler = StandardScaler(), StandardScaler()
    xF_train = feature_scaler.fit_transform(xF_train)
    xF_val   = feature_scaler.transform(xF_val)
    xF_test  = feature_scaler.transform(xF_test)
    xE_train = embedding_scaler.fit_transform(xE_train)
    xE_val   = embedding_scaler.transform(xE_val)
    xE_test  = embedding_scaler.transform(xE_test)

    return (xF_train, xF_val, xF_test, xE_train, xE_val, xE_test,
            y_train, y_val, y_test, final_train_idx, final_val_idx, test_idx)


In [19]:
def convert_to_binary_classification(y, threshold=0.3):
    
    return (y >= threshold).astype(int)

def predict_and_evaluate_nn_fusion(
    model, model_name, embedding_name,
    x_features_train, x_features_val, x_features_test,
    x_embeddings_train, x_embeddings_val, x_embeddings_test,
    y_train, y_val, y_test,
    train_sequences=None, val_sequences=None, test_sequences=None,
    save_predictions=True, fold = None
):

    
    print(f"  training {model_name} with {embedding_name} embeddings...")
    
    # train model
    model.fit(
        x_features_train, x_embeddings_train, y_train,
        x_features_val, x_embeddings_val, y_val
    )


    # extract predictions
    y_pred_train = model.predict(x_features_train, x_embeddings_train)
    y_pred_val = model.predict(x_features_val, x_embeddings_val)
    y_pred_test = model.predict(x_features_test, x_embeddings_test)
    
    # extract probabilities for auc
    y_pred_proba_train = model.predict_proba(x_features_train, x_embeddings_train)[:, 1]
    y_pred_proba_val = model.predict_proba(x_features_val, x_embeddings_val)[:, 1]
    y_pred_proba_test = model.predict_proba(x_features_test, x_embeddings_test)[:, 1]
    
    # calculate metrics
    train_accuracy = round(accuracy_score(y_train, y_pred_train), 4)
    val_accuracy = round(accuracy_score(y_val, y_pred_val), 4)
    test_accuracy = round(accuracy_score(y_test, y_pred_test), 4)
    train_f1 = round(f1_score(y_train, y_pred_train), 4)
    val_f1 = round(f1_score(y_val, y_pred_val), 4)
    test_f1 = round(f1_score(y_test, y_pred_test), 4)
    train_precision = round(precision_score(y_train, y_pred_train), 4)
    val_precision = round(precision_score(y_val, y_pred_val), 4)
    test_precision = round(precision_score(y_test, y_pred_test), 4)
    train_recall = round(recall_score(y_train, y_pred_train), 4)
    val_recall = round(recall_score(y_val, y_pred_val), 4)
    test_recall = round(recall_score(y_test, y_pred_test), 4)
    train_auc = round(roc_auc_score(y_train, y_pred_proba_train), 4)
    val_auc = round(roc_auc_score(y_val, y_pred_proba_val), 4)
    test_auc = round(roc_auc_score(y_test, y_pred_proba_test), 4)

    # compute per-sample losses
    train_losses_per_sample = model.get_per_sample_losses(
        x_features_train, x_embeddings_train, y_train
    )
    val_losses_per_sample = model.get_per_sample_losses(
        x_features_val, x_embeddings_val, y_val
    )
    test_losses_per_sample = model.get_per_sample_losses(
        x_features_test, x_embeddings_test, y_test
    )

    # key by embedding + fold so folds don't overwrite each other
    result_key = f"{embedding_name}_fold{fold}"
    
    # store predictions 
    if save_predictions:
        model_key = f"{model_name}_{result_key}"
        trained_models_dict[model_key] = {
            'model': model, 
            'model_name': model_name,
            'embedding_name': embedding_name,
            'feature_scaler': None,
            'embedding_scaler': None
        }
        all_predictions_dict[result_key] = {
            'y_train': y_train,
            'y_val': y_val,
            'y_test': y_test,
            'y_pred_train': y_pred_train,
            'y_pred_val': y_pred_val,
            'y_pred_test': y_pred_test,
            'y_pred_proba_train': y_pred_proba_train,
            'y_pred_proba_val': y_pred_proba_val,
            'y_pred_proba_test': y_pred_proba_test,
            'train_indices': None,
            'test_indices': None,
            'x_features_train': x_features_train,
            'x_features_val': x_features_val,
            'x_features_test': x_features_test,
            'x_embeddings_train': x_embeddings_train,
            'x_embeddings_val': x_embeddings_val,
            'x_embeddings_test': x_embeddings_test,
            'train_sequences': train_sequences,
            'val_sequences': val_sequences,
            'test_sequences': test_sequences,
            'train_indices': None,
            'val_indices': None,
            'test_indices': None,
            'train_original_target': None,
            'val_original_target': None,
            'test_original_target': None,
            'train_losses_per_sample': train_losses_per_sample,
            'val_losses_per_sample': val_losses_per_sample,
            'test_losses_per_sample': test_losses_per_sample
        }
        
    # store results
    clf_list.append({
        'model': model_name, 
        'embedding': embedding_name, 
        'fusion_type': 'intermediate_nn',
        'train_accuracy': train_accuracy, 
        'val_accuracy': val_accuracy,
        'test_accuracy': test_accuracy,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'test_f1': test_f1,
        'train_precision': train_precision,
        'val_precision': val_precision,
        'test_precision': test_precision,
        'train_recall': train_recall,
        'val_recall': val_recall,
        'test_recall': test_recall,
        'train_auc': train_auc,
        'val_auc': val_auc,
        'test_auc': test_auc,
        'feature_dim': x_features_train.shape[1],
        'embedding_dim': x_embeddings_train.shape[1],
        'fold': fold
    })
    
    print(f"  finished evaluating {model_name} with {embedding_name} embeddings")
    print(f"    test accuracy: {test_accuracy:.4f}, test auc: {test_auc:.4f}")
    
    return model

In [20]:
from sklearn.model_selection import GroupKFold

n_splits = 10
gkf = GroupKFold(n_splits=n_splits)

print("starting neural network intermediate fusion (grouped cross-validation by protein_id)")
print(f"results will be saved to: {output_dir}")

for key, value in embedding_paths.items():
    print(f"\n{'='*60}")
    print(f"processing {key} embeddings...")
    print('='*60)

    try:
        # load embedding data as df
        embedding_df = load_embedding_data(value)

        # extract features for both modalities, plus protein_id groups
        x_features, x_embeddings, y, groups, subset_df = extract_modality_features(
            embedding_df, target_col="medium_GFP_fraction_cells_with_condensates"
        )

        # convert regression target to binary classification
        y_binary = convert_to_binary_classification(y, threshold=0.3)

        for fold, (train_idx, test_idx) in enumerate(gkf.split(x_features, y_binary, groups=groups), start=1):
            print(f"\n  --- fold {fold}/{n_splits} ---")

            try:
                # split data and get train/val/test for both modalities, grouped by protein_id
                (x_features_train, x_features_val, x_features_test,
                 x_embeddings_train, x_embeddings_val, x_embeddings_test,
                 y_train, y_val, y_test,
                 final_train_idx, final_val_idx, final_test_idx) = prepare_data_modalities_for_fold(
                    x_features, x_embeddings, y_binary, groups, train_idx, test_idx
                )

                train_original_target = y[final_train_idx]
                val_original_target   = y[final_val_idx]
                test_original_target  = y[final_test_idx]

                train_sequences = subset_df.iloc[final_train_idx]['seq'].values
                val_sequences   = subset_df.iloc[final_val_idx]['seq'].values
                test_sequences  = subset_df.iloc[final_test_idx]['seq'].values

                # validation check
                print(f"    data shapes:")
                print(f"      x_features: {x_features_train.shape[1]} features, {x_features_train.shape[0]} samples")
                print(f"      x_embeddings: {x_embeddings_train.shape[1]} dimensions, {x_embeddings_train.shape[0]} samples")
                print(f"      class distribution - train: {np.unique(y_train, return_counts=True)}")
                print(f"      class distribution - test: {np.unique(y_test, return_counts=True)}")

                # initialize neural network intermediate fusion classifier
                # adjust learning rate and epochs based on embedding dimension
                epochs = 50 if x_embeddings_train.shape[1] < 1000 else 30  # fewer epochs for very high-dimensional embeddings

                fusion_clf_nn = intermediateFusionNNClassifier(
                    feature_dim=x_features_train.shape[1],
                    embedding_dim=x_embeddings_train.shape[1],
                    lr=0.001,
                    epochs=epochs,
                    batch_size=32,
                    dropout_rate=0.4,
                    use_cross_attention=True,
                    patience=5,
                    min_delta=0.005
                )

                # train and evaluate neural network fusion model
                trained_model = predict_and_evaluate_nn_fusion(
                    fusion_clf_nn, 'intermediate_fusion_nn', key,
                    x_features_train, x_features_val, x_features_test,
                    x_embeddings_train, x_embeddings_val, x_embeddings_test,
                    y_train, y_val, y_test,
                    train_sequences=train_sequences,
                    val_sequences=val_sequences,
                    test_sequences=test_sequences,
                    save_predictions=True,
                    fold=fold
                )

                # update all_predictions_dict for this embedding/fold combo
                all_predictions_dict[f"{key}_fold{fold}"].update({
                    'train_indices': final_train_idx.tolist(),
                    'val_indices': final_val_idx.tolist(),
                    'test_indices': final_test_idx.tolist(),
                    'train_original_target': train_original_target,
                    'val_original_target': val_original_target,
                    'test_original_target': test_original_target,
                    'original_data_df': subset_df,
                    'embedding_df': embedding_df
                })

            except Exception as fold_e:
                print(f"    error processing {key} fold {fold}: {str(fold_e)}")
                continue

    except Exception as e:
        print(f"  error processing {key}: {str(e)}")
        continue

starting neural network intermediate fusion (grouped cross-validation by protein_id)
results will be saved to: model_results/nn_fusion_clf_results_cross_val/20260707_111545

processing seqvec embeddings...

  --- fold 1/10 ---
    data shapes:
      x_features: 105 features, 2508 samples
      x_embeddings: 1024 dimensions, 2508 samples
      class distribution - train: (array([0, 1]), array([1386, 1122]))
      class distribution - test: (array([0, 1]), array([143, 226]))
  training intermediate_fusion_nn with seqvec embeddings...
  epoch 10/30, train loss: 0.3522, val loss: 0.3104, patience: 0/5
  epoch 20/30, train loss: 0.2680, val loss: 0.3130, patience: 2/5
  Early stopping triggered at epoch 24
  Restoring best model with validation loss: 0.2802
  finished evaluating intermediate_fusion_nn with seqvec embeddings
    test accuracy: 0.8862, test auc: 0.9584

  --- fold 2/10 ---
    data shapes:
      x_features: 105 features, 2977 samples
      x_embeddings: 1024 dimensions, 2977 

In [21]:
# save per-fold results dataframe
clf_df = pd.DataFrame(clf_list)
results_file = f"{output_dir}/intermediate_fusion_nn_results_per_fold.csv"
clf_df.to_csv(results_file, index=False)

# aggregate across folds: mean/std per model+embedding
metric_cols = [
    'train_accuracy', 'val_accuracy', 'test_accuracy',
    'train_f1', 'val_f1', 'test_f1',
    'train_precision', 'val_precision', 'test_precision',
    'train_recall', 'val_recall', 'test_recall',
    'train_auc', 'val_auc', 'test_auc'
]

agg_df = clf_df.groupby(['model', 'embedding']).agg(
    {col: ['mean', 'std'] for col in metric_cols}
)
agg_df.columns = ['_'.join(c) for c in agg_df.columns]
agg_df = agg_df.reset_index()

agg_file = f"{output_dir}/intermediate_fusion_nn_results_aggregated.csv"
agg_df.to_csv(agg_file, index=False)

# save detailed summary
summary_file = f"{output_dir}/summary.txt"
with open(summary_file, 'w') as f:
    f.write(f"neural network intermediate fusion experiment (grouped cross-validation by protein_id)\n")
    f.write(f"timestamp: {timestamp}\n")
    f.write(f"number of embeddings tested: {len(embedding_paths)}\n")
    f.write(f"number of folds: {n_splits}\n")
    f.write(f"feature dimension: {len(feature_cols)}\n")
    f.write(f"target: medium_gfp_fraction_cells_with_condensates (binary @ 0.3)\n")
    f.write(f"\nresults summary (mean ± std across {n_splits} folds):\n")
    f.write('='*50 + '\n')

    for _, row in agg_df.iterrows():
        f.write(f"\nmodel: {row['model']} with {row['embedding']}\n")
        f.write(f"  test accuracy: {row['test_accuracy_mean']:.4f} ± {row['test_accuracy_std']:.4f}\n")
        f.write(f"  test auc: {row['test_auc_mean']:.4f} ± {row['test_auc_std']:.4f}\n")
        f.write(f"  test f1: {row['test_f1_mean']:.4f} ± {row['test_f1_std']:.4f}\n")

print(f"results saved to {output_dir}")
print(f"  - per-fold results: {results_file}")
print(f"  - aggregated results: {agg_file}")
print(f"  - experiment summary: {summary_file}")

results saved to model_results/nn_fusion_clf_results_cross_val/20260707_111545
  - per-fold results: model_results/nn_fusion_clf_results_cross_val/20260707_111545/intermediate_fusion_nn_results_per_fold.csv
  - aggregated results: model_results/nn_fusion_clf_results_cross_val/20260707_111545/intermediate_fusion_nn_results_aggregated.csv
  - experiment summary: model_results/nn_fusion_clf_results_cross_val/20260707_111545/summary.txt


In [22]:
print(f"total fold-runs evaluated: {len(clf_list)}")
print(f"total embedding/model combos evaluated: {len(agg_df)}")

if len(agg_df) > 0:
    best_idx = agg_df['test_auc_mean'].idxmax()
    best_model = agg_df.loc[best_idx]

    print(f"\nbest performing model (by mean test AUC across folds):")
    print(f"  embedding: {best_model['embedding']}")
    print(f"  test auc: {best_model['test_auc_mean']:.4f} ± {best_model['test_auc_std']:.4f}")
    print(f"  test accuracy: {best_model['test_accuracy_mean']:.4f} ± {best_model['test_accuracy_std']:.4f}")
    print(f"  test f1: {best_model['test_f1_mean']:.4f} ± {best_model['test_f1_std']:.4f}")

print(f"\nall results have been saved to: {output_dir}")

total fold-runs evaluated: 80
total embedding/model combos evaluated: 8

best performing model (by mean test AUC across folds):
  embedding: seqvec
  test auc: 0.9362 ± 0.0255
  test accuracy: 0.8655 ± 0.0432
  test f1: 0.8286 ± 0.0613

all results have been saved to: model_results/nn_fusion_clf_results_cross_val/20260707_111545


In [23]:
# save the all_predictions_dict to a file
import pickle
import json

if (save_predictions_pkl == True):
    # try to save as pickle (preserves complex objects)
    try:
        predictions_pickle_file = f"{output_dir}/all_predictions_dict.pkl"
        with open(predictions_pickle_file, 'wb') as f:
            pickle.dump(all_predictions_dict, f)
        print(f"Saved predictions dictionary (pickle): {predictions_pickle_file}")
    except Exception as e:
        print(f"Could not save pickle file: {e}")
    
    # also save a simplified JSON version (for easier inspection)
    try:
        # create a simplified version for JSON serialization
        simplified_predictions = {}
        for embedding_name, data in all_predictions_dict.items():
            simplified_predictions[embedding_name] = {
                'y_train_shape': data['y_train'].shape if hasattr(data['y_train'], 'shape') else str(type(data['y_train'])),
                'y_test_shape': data['y_test'].shape if hasattr(data['y_test'], 'shape') else str(type(data['y_test'])),
                'y_pred_train_shape': data['y_pred_train'].shape if hasattr(data['y_pred_train'], 'shape') else str(type(data['y_pred_train'])),
                'y_pred_test_shape': data['y_pred_test'].shape if hasattr(data['y_pred_test'], 'shape') else str(type(data['y_pred_test'])),
                'train_sequences_count': len(data['train_sequences']) if data['train_sequences'] is not None else 0,
                'test_sequences_count': len(data['test_sequences']) if data['test_sequences'] is not None else 0,
                'train_indices_count': len(data['train_indices']) if data['train_indices'] is not None else 0,
                'test_indices_count': len(data['test_indices']) if data['test_indices'] is not None else 0
            }
        
        predictions_json_file = f"{output_dir}/all_predictions_summary.json"
        with open(predictions_json_file, 'w') as f:
            json.dump(simplified_predictions, f, indent=2)
        print(f"Saved predictions summary (JSON): {predictions_json_file}")
    
    except Exception as e:
        print(f"Could not save JSON file: {e}")